# Hands-on: can a team of models beat a single model?

**SDS ML Workshop, about 30 minutes**

You will predict how much sleep debt a person has from their evening habits. First you fit four
single models, then you combine them into a **stacked ensemble** and see whether the team beats one
model on its own.

**How to use this notebook**
1. Run the cells from top to bottom (Shift + Enter).
2. Only edit cells marked **YOUR TURN**. Everything else is already set up.
3. Before handing in, click **Restart** and then **Run All** in the notebook toolbar to check
   everything still runs.

**Rules**
- Judge your models only with the CV score printed in this notebook.
- Do not touch the test split. It is kept aside for the facilitator.
- No external data.

In [ ]:
# Install the packages this notebook needs into the kernel you selected
%pip install -q -r requirements.txt

## 1. Setup (do not edit)

**The task:** predict `sleep_debt_category`, one of Optimal Recovery, Mild Deficit, Moderate Debt or
Severe Sleep Debt.

**The score:** macro F1, an average score across the 4 categories where each counts equally, so rare
ones like Severe Sleep Debt matter. 1.0 is perfect.

**Why some columns are removed:** the category is a simple rule on `total_sleep_hours` and
`next_day_fatigue_score`. Those two columns alone give about 100% accuracy, so keeping them would turn
the challenge into copying the answer. We also drop `user_id`, which is just a label. The data is
probably synthetic, so treat the results as practice, not science.

**How we score fairly:** cross-validation (CV) splits the training data into 5 parts, trains on 4 and
tests on the 5th, five times over. You get an average score (mean) and how much it wobbles (std).
Every model is tested on the same 5 parts, so the comparison is fair.

In [ ]:
# Imports and settings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

SEED = 42

In [ ]:
# Load the data, drop the leaky columns, split 80/20 and set up the scoring tool
DATA_FILE = "bedtime_screentime_sleep_debt.csv"
found = next((p for p in [os.path.join("data", DATA_FILE), DATA_FILE] if os.path.exists(p)), None)
if found:
    df = pd.read_csv(found)
else:
    import kagglehub  # only needed when the CSV is not in data/ or next to this notebook
    folder = kagglehub.dataset_download("samartalwar/sleep-debt-and-screen-time-late-night-phone-habits")
    df = pd.read_csv(os.path.join(folder, DATA_FILE))

CLASS_NAMES = ["Optimal Recovery", "Mild Deficit", "Moderate Debt", "Severe Sleep Debt"]
LABELS = {name: i for i, name in enumerate(CLASS_NAMES)}  # XGBoost needs numbers, not words
NUMERIC = ["age", "bedtime_phone_minutes", "screen_brightness_pct", "blue_light_filter_active",
           "caffeine_post_5pm_mg", "physical_activity_min", "sleep_latency_min",
           "deep_sleep_pct", "rem_sleep_pct", "morning_alarm_snoozes"]
CATEGORICAL = ["gender", "occupation_type", "chronotype", "primary_bedtime_app"]

df = df.drop(columns=["user_id", "total_sleep_hours", "next_day_fatigue_score"])
X = df[NUMERIC + CATEGORICAL]
y = df["sleep_debt_category"].map(LABELS)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)


def pipe(model):
    # Scale the numbers and one-hot the categories, then fit the model
    prep = ColumnTransformer([("num", StandardScaler(), NUMERIC),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return Pipeline([("prep", prep), ("model", model)])


def cv_score(model):
    # 5-fold CV macro F1 on the TRAIN split only. Returns the 5 fold scores
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    return cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)


print(f"Training rows: {len(X_train)}, test rows kept aside: {len(X_test)}")
print(y_train.map(dict(enumerate(CLASS_NAMES))).value_counts(normalize=True).round(2))

## 2. The baseline (do not edit)

The baseline is a Random Forest with 100 trees, already a strong single model. Its CV score is the
bar to beat. How much its score wobbles between the 5 folds is the **noise level**: a gain smaller
than that could just be luck.

In [ ]:
# Build the baseline and score it with CV
baseline = pipe(RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))
baseline_folds = cv_score(baseline)
NOISE = baseline_folds.std()  # how much the baseline wobbles between folds
print(f"Baseline (Random Forest): CV macro F1 = {baseline_folds.mean():.3f}, noise level = {NOISE:.3f}")

## 3. Part A: fit four single models

Each cell below already runs as it is. **YOUR TURN:** change at most one setting per model, then
rerun the scoring cell at the end of Part A.

**KNN:** asks the 5 most similar people in the training data and takes a vote.

In [ ]:
# YOUR TURN: KNN. Try n_neighbors between 3 and 40
knn = pipe(KNeighborsClassifier(n_neighbors=5))

**Decision Tree:** asks a chain of yes or no questions ("caffeine after 5pm over 100 mg?") and
follows the answers to a category. `max_depth` is how many questions it may ask in a row.

In [ ]:
# YOUR TURN: Decision Tree. Try max_depth between 3 and 8
tree = pipe(DecisionTreeClassifier(max_depth=5, random_state=SEED))

**Random Forest:** grows many different trees on random slices of the data, and they vote.

In [ ]:
# YOUR TURN: Random Forest. Try n_estimators between 50 and 200, or max_depth between 5 and 15
forest = pipe(RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))

**XGBoost:** builds small trees one after another, each one fixing the mistakes of the trees before it.

In [ ]:
# YOUR TURN: XGBoost. Try n_estimators 50 to 200, max_depth 2 to 6, or learning_rate 0.02 to 0.2
xgb = pipe(XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.9,
                         colsample_bytree=0.9, random_state=SEED, n_jobs=-1))

In [ ]:
# Score all four models with CV and compare them (about 30 seconds)
models = {"KNN": knn, "Decision Tree": tree, "Random Forest": forest, "XGBoost": xgb}
folds = {name: cv_score(m) for name, m in models.items()}

results = pd.DataFrame({name: {"CV mean": s.mean(), "CV std": s.std()} for name, s in folds.items()}).T
results = results.sort_values("CV mean", ascending=False)
results["vs baseline"] = results["CV mean"] - baseline_folds.mean()
display(results.round(3))

ax = results["CV mean"].iloc[::-1].plot.barh(xerr=results["CV std"].iloc[::-1], color="C0", figsize=(7, 3))
ax.axvline(baseline_folds.mean(), ls="--", color="grey", label="baseline (Random Forest)")
ax.set_xlim(0.5, 0.8)
ax.set_xlabel("CV macro F1 (higher is better)")
ax.legend()
plt.show()

**Think about it:** which single model is best? Does any single model beat the baseline by more
than the noise level?

## 4. Part B: build a stacked ensemble

Each model gives its opinion: a probability for each category. A final simple model, the
**meta-learner**, learns how much to trust each one.

It learns from predictions the models made on data they did not train on (called out-of-fold).
That stops it over-trusting a model that just memorised the training data.

**YOUR TURN:** choose which models go into the stack. Start with all four.

In [ ]:
# YOUR TURN: delete a line to drop a model from the stack
MODELS_TO_STACK = {
    "Decision Tree": tree,
    "Random Forest": forest,
    "XGBoost": xgb,
    "KNN": knn,
}

Scoring the stack takes about 1 to 2 minutes because every model is trained many times. A quiet
cell is normal.

In [ ]:
# Build the stack and score it with CV
def make_stack(models_to_stack):
    return StackingClassifier(estimators=list(models_to_stack.items()),
                              final_estimator=LogisticRegression(max_iter=2000),
                              cv=5, stack_method="predict_proba", n_jobs=-1)


stack = make_stack(MODELS_TO_STACK)
stack_folds = cv_score(stack)
print(f"Stack of {len(MODELS_TO_STACK)} models: CV macro F1 = {stack_folds.mean():.3f} +/- {stack_folds.std():.3f}")

## 5. Part C: compare and conclude

A gain only counts if it is bigger than the **noise level**, the baseline's own wobble between folds
(printed above the table). It is measured from the data, not fixed in advance.

**folds won** shows on how many of the 5 folds a model scored higher than the baseline. A model that
wins 4 or 5 folds is consistently better, even if the average gain is small.

In [ ]:
# One table: baseline, the four single models and the stack
all_folds = {"Baseline": baseline_folds, **folds, "Stack": stack_folds}
best_single = results["CV mean"].max()


def verdict(name, s):
    gain = s.mean() - baseline_folds.mean()
    if name == "Baseline":
        return "The score to beat"
    if gain > NOISE:
        if name == "Stack" and s.mean() - best_single > NOISE:
            return "Beats the baseline and the best single model"
        return "Beats the baseline"
    if gain < -NOISE:
        return "Worse than the baseline"
    return "About the same as the baseline (within noise)"


table = pd.DataFrame([{"model": n, "CV mean": round(s.mean(), 3),
                       "vs baseline": round(s.mean() - baseline_folds.mean(), 3),
                       "folds won": "-" if n == "Baseline" else f"{(s > baseline_folds).sum()}/5",
                       "verdict": verdict(n, s)} for n, s in all_folds.items()]).set_index("model")
print(f"Noise level: {NOISE:.3f}. A gain must be bigger than this to count.")
display(table)

**Question 1.** Did the stack beat your best single model, or only the baseline? Why might the gain
be small?

**Your answer:**

**Question 2.** Run the cell below. It fits the stack once on the training data and shows how much
the meta-learner trusts each model (bigger = trusted more). Which model does it trust most?

In [ ]:
# Fit the stack once on the training data and show one trust weight per model
def show_weights(fitted_stack):
    coef = np.abs(fitted_stack.final_estimator_.coef_)  # rows: categories, columns: 4 per model
    names = [n for n, _ in fitted_stack.estimators]
    per_model = coef.reshape(coef.shape[0], len(names), -1).mean(axis=(0, 2))
    print(pd.Series(per_model, index=names).sort_values(ascending=False).round(3).to_string())


stack.fit(X_train, y_train)
show_weights(stack)

**Your answer:**

**Question 3.** Remove one model from `MODELS_TO_STACK` in Part B, rerun from there down, and
compare. What happened to the stack's score?

**Your answer:**